# goals term — E[goals] one gameweek ahead

**Render-not-decide.** This notebook is a *re-runnable view* of the `goals` term, **not** the record of its numbers — the frozen record is [predictive-phase2-component-model.md](../../../docs/studies/results/predictive-phase2-component-model.md). All logic lives in `model/terms/goals/`; the notebook only calls it and plots.

Population: `minutes > 0`, DGW excluded, GW > 3, expanding walk-forward, **conditional on appearance** (see `ASSUMPTIONS.md`).

## Setup
> Load the mart, then instantiate the term. `GoalsModel(minimal)` is the mechanistic bar *and* the shipped design today; `GoalsTerm` scores it against its own lagged-goals baseline.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.goals import GoalsModel, GoalsTerm
from model.terms.goals.spec import GOALS_POOL

mart = load_mart()
term = GoalsTerm(GoalsModel(variant="minimal"))
GOALS_POOL.minimal, [f.name for f in GOALS_POOL.candidates]

## Pre-fit assumptions
> Before fitting: is Poisson justified (material dispersion ~1) and is the effect learnable at this N (detectability floor)? See `ASSUMPTIONS.md` §1, §5.

In [ ]:
report = term.model.check_assumptions(GoalsModel.population(mart))
print(f"family_ok={report.family_ok}  detectable={report.detectable}  n_train={report.n_train}")
report.dispersion

## Gate — E[goals] vs the term's own lagged-goals baseline
> Per-term level (spec §5): does the *signal-based* model out-rank a player's naive goals history, within position? A pass at the rankable attacking positions (DEF/MID) is the goals bet.

In [ ]:
gate = term.validate(mart)
display(gate.table)

ax = gate.table.set_index("position")[["baseline", "e_goals"]].plot.bar(figsize=(7, 3.5))
ax.set_ylabel("within-position Spearman"); ax.set_title("goals: model vs own baseline"); plt.tight_layout()

## Diagnose — residuals + ablation
> Post-gate (spec §4 stage 5). Residuals: the worst-missed player-GWs. Ablation: drop each feature, re-score — the *measured* contribution of `xgi_roll3` vs `minutes_roll3`.

In [ ]:
diag = term.diagnose(mart)
display(diag.ablation)
display(diag.residuals.head(10))